In [14]:
import sqlite3

paths = [
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq128_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq512_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq4096_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq128_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq512_bs4_fp32.sqlite",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq1024_bs4_fp32.sqlite"
]

config_path_mapping = {
    path.split('/')[-1].split(".sqlite")[0]: path for path in paths
}


In [59]:
config_path_mapping

{'small_seq128_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq128_bs4_fp32.sqlite',
 'small_seq512_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq512_bs4_fp32.sqlite',
 'small_seq4096_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq4096_bs4_fp32.sqlite',
 'xl_seq128_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq128_bs4_fp32.sqlite',
 'xl_seq512_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq512_bs4_fp32.sqlite',
 'xl_seq1024_bs4_fp32': '../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq1024_bs4_fp32.sqlite'}

In [63]:
import statistics

results_df_rows = []

for config, path in config_path_mapping.items():
    connection = sqlite3.connect(path)
    cursor = connection.cursor()

    forward_sql_code = """
    SELECT
        COALESCE(n.text, s.value) AS name,
        n.start,
        n.end
    FROM NVTX_EVENTS AS n
    LEFT JOIN StringIds AS s ON n.textId = s.id
    WHERE COALESCE(n.text, s.value) LIKE 'step_%/forward'
    ORDER BY n.start;
    """

    cursor.execute(forward_sql_code)
    result = cursor.fetchall()

    cpu_forward_times_ms = []

    for res in result:
        cpu_forward_times_ms.append((res[2] - res[1]) / 10 ** 6)

    print(f"time for forward: {statistics.mean(cpu_forward_times_ms)} +- {statistics.stdev(cpu_forward_times_ms)} ms")

    backward_sql_code = """
    SELECT
        COALESCE(n.text, s.value) AS name,
        n.start,
        n.end
    FROM NVTX_EVENTS AS n
    LEFT JOIN StringIds AS s ON n.textId = s.id
    WHERE COALESCE(n.text, s.value) LIKE 'torch.autograd.backward'
    ORDER BY n.start;
    """

    cursor.execute(backward_sql_code)
    result = cursor.fetchall()
    
    cpu_backward_times_ms = []

    for res in result:
        cpu_backward_times_ms.append((res[2] - res[1]) / 10 ** 6)

    print(f"time for backward: {statistics.mean(cpu_backward_times_ms)} +- {statistics.stdev(cpu_backward_times_ms)} ms")

    cpu_fb_times_ms = []

    for forward_time, backward_time in zip(cpu_forward_times_ms, cpu_backward_times_ms):
        cpu_fb_times_ms.append(forward_time + backward_time)

    print(f"time for forward + backward: {statistics.mean(cpu_fb_times_ms)} +- {statistics.stdev(cpu_fb_times_ms)} ms")

    optimizer_sql_code = """
    SELECT
        COALESCE(n.text, s.value) AS name,
        n.start,
        n.end
    FROM NVTX_EVENTS AS n
    LEFT JOIN StringIds AS s ON n.textId = s.id
    WHERE COALESCE(n.text, s.value) LIKE 'AdamW.step'
    ORDER BY n.start;
    """

    cursor.execute(optimizer_sql_code)
    result = cursor.fetchall()

    cpu_optimizer_times_ms = []

    for res in result:
        cpu_optimizer_times_ms.append((res[2] - res[1]) / 10 ** 6)

    print(f"time for optimizer: {statistics.mean(cpu_optimizer_times_ms)} +- {statistics.stdev(cpu_optimizer_times_ms)} ms")

    cpu_fbo_times_ms = []

    for forward_time, backward_time, optimizer_time in zip(cpu_forward_times_ms, cpu_backward_times_ms, cpu_optimizer_times_ms):
        cpu_fbo_times_ms.append(forward_time + backward_time + optimizer_time)

    print(f"time for forward + backward + optimizer: {statistics.mean(cpu_fbo_times_ms)} +- {statistics.stdev(cpu_fbo_times_ms)} ms")

    result_df_row = {
        "config": config,
        "forward_times_mean": statistics.mean(cpu_forward_times_ms),
        "forward_times_std": statistics.stdev(cpu_forward_times_ms),
        "backward_times_mean": statistics.mean(cpu_backward_times_ms),
        "backward_times_std": statistics.stdev(cpu_backward_times_ms),
        "optimizer_times_mean": statistics.mean(cpu_optimizer_times_ms),
        "optimizer_times_std": statistics.stdev(cpu_optimizer_times_ms),
        "forwad+backward_times_mean": statistics.mean(cpu_fb_times_ms),
        "forwad+backward_times_std": statistics.stdev(cpu_fb_times_ms),
        "forward+backward+optimizer_times_mean": statistics.mean(cpu_fbo_times_ms),
        "forward+backward+optimizer_times_std": statistics.stdev(cpu_fbo_times_ms),
    }

    results_df_rows.append(result_df_row)

time for forward: 31.0429569 +- 12.089597007699155 ms
time for backward: 51.9928441 +- 26.62416685410959 ms
time for forward + backward: 83.03580099999999 +- 29.51705663172357 ms
time for optimizer: 22.9508702 +- 12.563189602491734 ms
time for forward + backward + optimizer: 105.9866712 +- 41.38080361158804 ms
time for forward: 31.2574328 +- 4.369274863807973 ms
time for backward: 53.863613199999996 +- 22.71498409598575 ms
time for forward + backward: 85.12104599999999 +- 23.171400421599735 ms
time for optimizer: 20.8335266 +- 3.115037963336213 ms
time for forward + backward + optimizer: 105.95457259999999 +- 25.838086582158994 ms
time for forward: 264.9648831 +- 6.064642276465643 ms
time for backward: 285.5206424 +- 104.86958558867666 ms
time for forward + backward: 550.4855255 +- 110.85142250126346 ms
time for optimizer: 282.897028 +- 90.19706620247298 ms
time for forward + backward + optimizer: 833.3825535000001 +- 20.666347145011223 ms
time for forward: 143.5578278 +- 3.35098516706

In [66]:
import pandas as pd

results_df = pd.DataFrame(results_df_rows)
results_df.head()

results_df.to_csv("../benches/nsys_runs/assignment_runs.csv", index=False)

In [69]:
results_df

,config,forward_times_mean,forward_times_std,backward_times_mean,backward_times_std,optimizer_times_mean,optimizer_times_std,forwad+backward_times_mean,forwad+backward_times_std,forward+backward+optimizer_times_mean,forward+backward+optimizer_times_std
0,small_seq128_bs4_fp32,31.042957,12.089597,51.992844,26.624167,22.950870,12.563190,83.035801,29.517057,105.986671,41.380804
1,small_seq512_bs4_fp32,31.257433,4.369275,53.863613,22.714984,20.833527,3.115038,85.121046,23.171400,105.954573,25.838087
2,small_seq4096_bs4_fp32,264.964883,6.064642,285.520642,104.869586,282.897028,90.197066,550.485525,110.851423,833.382554,20.666347
3,xl_seq128_bs4_fp32,143.557828,3.350985,166.067098,55.405439,87.987998,19.107851,309.624926,54.042296,397.612923,71.144687
4,xl_seq512_bs4_fp32,317.691778,16.451605,456.718708,58.449301,188.869304,12.007225,774.410486,58.561036,963.279790,47.124305
5,xl_seq1024_bs4_fp32,602.919144,12.381229,939.449318,100.542163,302.117571,79.811720,1542.368462,101.267694,1844.486033,23.933857


# Task b - find most fat gpu kernels inside forward, and forward + backward

### reran and get data using 
`
nsys stats \
  --report nvtx_kern_sum \
  --format csv
`
### command

In [11]:
import pandas as pd
pd.set_option("display.max_colwidth", 75)


paths = [
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq128_bs4_fp32_nvtx_kern_sum.csv",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq512_bs4_fp32_nvtx_kern_sum.csv",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-1bf3b9ce/small_seq4096_bs4_fp32_nvtx_kern_sum.csv",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq128_bs4_fp32_nvtx_kern_sum.csv",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq512_bs4_fp32_nvtx_kern_sum.csv",
    "../benches/nsys_runs/20261018T141854Z-nsys-fp32-323a83fd/xl_seq1024_bs4_fp32_nvtx_kern_sum.csv"
]

In [12]:
config_fattest_kernels_mapping = {}

for path in paths:
    df = pd.read_csv(path)
    steps_ranges = []
    for idx in range(10):
        steps_ranges.append(df[df['NVTX Range'] == f":step_{idx}/forward"])
    
    forward_ranges = pd.concat(steps_ranges)
    fattest_kernels = forward_ranges.groupby("Kernel Name")[["Total Time (ns)", "Kern Inst"]].sum().sort_values(by="Total Time (ns)", ascending=False).reset_index()
    fattest_kernels["Total Time(ms)"] = fattest_kernels["Total Time (ns)"] / 10 ** 6
    del fattest_kernels["Total Time (ns)"]

    config = path.split('/')[-1].split("_bs4")[0]
    config_fattest_kernels_mapping[config] = fattest_kernels

In [13]:
for config in config_fattest_kernels_mapping:
    print(config)
    display(config_fattest_kernels_mapping[config][:5])

small_seq128


,Kernel Name,Kern Inst,Total Time(ms)
0,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_tnn_ali...,600,22.116193
1,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_tnn_ali...,240,14.367453
2,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",1210,3.245927
3,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",960,2.820553
4,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x128x16_1x1x1_3_tnn_al...,10,1.969636


small_seq512


,Kernel Name,Kern Inst,Total Time(ms)
0,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x128x16_1x1x1_3_tnn_al...,250,53.699046
1,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_tnn_ali...,600,53.111669
2,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",370,5.999146
3,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",1210,5.701963
4,void cutlass::Kernel2<cutlass_80_simt_sgemm_128x64_8x5_nn_align1>(T1::P...,120,5.674155


small_seq4096


...

,Kernel Name,Kern Inst,Total Time(ms)
0,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_tnn_ali...,1600,469.669784
1,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x128x16_1x1x1_3_tnn_al...,650,404.319388
2,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",3210,13.810398
3,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",2560,12.521655
4,"void at::native::vectorized_elementwise_kernel<(int)4, at::native::Bina...",1290,6.500979


xl_seq512


,Kernel Name,Kern Inst,Total Time(ms)
0,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_tnn_a...,650,1245.339287
1,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x64x16_1x1x1_3_tnn_al...,1280,660.291379
2,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_tnn_ali...,320,638.799020
3,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_32x64x16_1x1x1_3_nnn_ali...,320,48.641389
4,void cutlass::Kernel2<cutlass_80_simt_sgemm_128x64_8x5_nn_align1>(T1::P...,320,46.097521


xl_seq1024


,Kernel Name,Kern Inst,Total Time(ms)
0,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x64x16_1x1x1_3_tnn_al...,1600,2447.365485
1,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_tnn_a...,650,2232.494605
2,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_32x64x16_1x1x1_3_nnn_ali...,320,172.129145
3,void cutlass::Kernel2<cutlass_80_simt_sgemm_128x128_8x4_nn_align1>(T1::...,320,172.085892
4,"void at::native::elementwise_kernel<(int)128, (int)2, void at::native::...",970,158.272109


### Part 2 with backwards: had to rewrite some code - added torch.cuda.profiler start & stop to the code + rerun with --regime 1

In [15]:
from io import StringIO
from pathlib import Path
import pandas as pd


def read_nsys_stats(path, report="cuda_gpu_kern_sum"):
    lines = Path(path).read_text(encoding="utf-8").splitlines()

    start = next(
        (i + 1 for i, line in enumerate(lines)
         if line.startswith("Processing ") and f"{report}.py]" in line),
        None,
    )
    if start is None:
        raise ValueError(f"Report {report!r} not found in {path}")

    table = []
    for line in lines[start:]:
        if not line.strip():
            if table:
                break
            continue
        if line.startswith("Processing "):
            break
        table.append(line)

    return pd.read_csv(StringIO("\n".join(table)))

In [23]:
paths_regime_1 = [
    "../benches/nsys_runs/20261019T171553Z-nsys-fp32-62a55532/small_seq128_bs4_fp32.stats.txt",
    "../benches/nsys_runs/20261019T171553Z-nsys-fp32-62a55532/small_seq512_bs4_fp32.stats.txt",
    "../benches/nsys_runs/20261019T171553Z-nsys-fp32-62a55532/small_seq4096_bs4_fp32.stats.txt",
    "../benches/nsys_runs/20261019T171553Z-nsys-fp32-f5754d5b/xl_seq128_bs4_fp32.stats.txt",
    "../benches/nsys_runs/20261019T171553Z-nsys-fp32-f5754d5b/xl_seq512_bs4_fp32.stats.txt",
    "../benches/nsys_runs/20261019T171553Z-nsys-fp32-f5754d5b/xl_seq1024_bs4_fp32.stats.txt",
]

config_fattest_kernels_mapping_fb = {}

In [24]:
for path in paths_regime_1:
    
    df = read_nsys_stats(path)

    config = path.split('/')[-1].split("_bs4")[0]
    config_fattest_kernels_mapping_fb[config] = df[:5][["Time (%)", "Total Time (ns)", "Instances", "Name"]]

In [29]:
list(config_fattest_kernels_mapping_fb.keys())

['small_seq128',
 'small_seq512',
 'small_seq4096',
 'xl_seq128',
 'xl_seq512',
 'xl_seq1024']

In [30]:
for config in config_fattest_kernels_mapping_fb:
    print(config)
    display(config_fattest_kernels_mapping_fb[config])

small_seq128


,Time (%),Total Time (ns),Instances,Name
0,18.8,31993063,960,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_nnn_ali...
1,13.5,22981015,720,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_tnn_ali...
2,9.8,16701553,360,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x64x16_1x1x1_3_ntn_al...
3,8.4,14345872,240,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_tnn_ali...
4,5.8,9811340,720,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_ntn_ali...


small_seq512


,Time (%),Total Time (ns),Instances,Name
0,23.4,110578643,970,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_nnn_ali...
1,12.2,57875251,720,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_tnn_ali...
2,12.1,57281488,360,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_ntn_a...
3,11.4,53772247,250,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x128x16_1x1x1_3_tnn_al...
4,7.5,35442560,720,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_ntn_ali...


small_seq4096


...

,Time (%),Total Time (ns),Instances,Name
0,17.3,468422099,1600,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_tnn_ali...
1,17.3,467898371,970,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_ntn_a...
2,15.3,413966658,650,void cutlass::Kernel2<cutlass_80_simt_sgemm_128x64_8x5_nn_align1>(T1::P...
3,15.0,406234983,970,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x128x16_1x1x1_3_tnn_al...
4,8.3,223518027,1280,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x32x16_1x1x1_3_nnn_ali...


xl_seq512


,Time (%),Total Time (ns),Instances,Name
0,21.5,1867836112,1930,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_nnn_ali...
1,19.6,1708958779,970,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_ntn_a...
2,14.3,1243795217,650,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_tnn_a...
3,8.5,735868897,1600,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x64x16_1x1x1_3_tnn_al...
4,7.3,638179680,320,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_tnn_ali...


xl_seq1024


,Time (%),Total Time (ns),Instances,Name
0,19.6,3435970532,1930,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x64x16_1x1x1_3_nnn_al...
1,19.2,3367679244,970,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_ntn_a...
2,13.9,2443219923,1600,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x64x16_1x1x1_3_tnn_al...
3,13.8,2413931250,970,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_128x128x16_1x1x1_3_tnn_a...
4,6.7,1165306862,1280,cutlass3x_sm100_simt_sgemm_f32_f32_f32_f32_f32_64x64x16_1x1x1_3_ntn_ali...
